# Vectorstores and Embeddings (v2 — corrected chunking)

Builds the vectorstore from `document_splits_v2.json` (produced by `03_document_splitting_v2.ipynb`), NOT the original `document_splits.json` that backs the currently-live app. This is a separate, parallel build — it does not touch or replace the live vectorstore until it's deliberately swapped in afterward.

What changed in the input data (full detail in `03_document_splitting_v2.ipynb`):
- Fixed article-boundary detection for lloc (previously mislabeled ~161 documents' worth of chunks by confusing mid-sentence citations for real headers).
- Fixed multiple dash/formatting variants that caused several lloc documents — including the Constitution itself — to lose most of their own text (Constitution: 35% -> 93%+ coverage).
- sjc/ccb judgments are now always one whole chunk per case, never fragmented.
- 56 exact-duplicate sjc case records removed.

Total chunk count: 25,738 (lloc 16,644 + sjc 9,001 + ccb 93), vs. 25,286 in the current live vectorstore.


In [ ]:
!pip install -q langchain-huggingface langchain-chroma langchain-core langchain-text-splitters langchain-community langchain-classic langchain-groq sentence-transformers transformers chromadb panel param

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Upload document_splits_v2.json to your Google Drive root (My Drive) first, via drive.google.com
# in a browser — NOT through this cell. Reading straight from Drive means if this runtime
# disconnects during the long embedding step below, you just re-mount and continue — no need to
# re-upload a 168MB file through the browser file-picker again.


In [ ]:
import json
from langchain_core.documents import Document

DRIVE_JSON_PATH = "/content/drive/MyDrive/document_splits_v2.json"

with open(DRIVE_JSON_PATH, encoding="utf-8") as f:
    records = json.load(f)

splits = [Document(page_content=r["page_content"], metadata=r["metadata"]) for r in records]
print(f"Loaded {len(splits)} chunks")


## Embeddings

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

#embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

In [ ]:
sentence1 = "المستاجر ملزم بدفع الاجرة في موعدها"
sentence2 = "يجب على المستاجر سداد بدل الايجار في الوقت المحدد"
sentence3 = "الطقس اليوم ممطر وبارد"

embedding1 = embedding.embed_query(sentence1)
embedding2 = embedding.embed_query(sentence2)
embedding3 = embedding.embed_query(sentence3)

In [ ]:
import numpy as np

print("sentence1 vs sentence2 (both about rent payment):", np.dot(embedding1, embedding2))
print("sentence1 vs sentence3 (unrelated):              ", np.dot(embedding1, embedding3))

## Vectorstore — Chroma, persisted locally

In [ ]:
from langchain_chroma import Chroma

persist_directory = "/content/chroma_v2"

embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"batch_size": 8},  # caps how many texts get embedded together at once
)

BATCH = 50
vectordb = None
for i in range(0, len(splits), BATCH):
    batch = splits[i:i + BATCH]
    if vectordb is None:
        vectordb = Chroma.from_documents(documents=batch, embedding=embedding, persist_directory=persist_directory)
    else:
        vectordb.add_documents(batch)
    print(f"{min(i + BATCH, len(splits))}/{len(splits)} embedded", flush=True)

print("DONE. Collection count:", vectordb._collection.count())


In [ ]:
print("Total chunks in vectorstore:", vectordb._collection.count())
print("Expected:", len(splits))

In [ ]:
#from langchain_chroma import Chroma

#persist_directory = "../data/chroma/"

#vectordb = Chroma.from_documents(
 #   documents=splits,
  #  embedding=embedding,
   # persist_directory=persist_directory,
#)
#print(vectordb._collection.count())

### Similarity search sanity check

In [ ]:
question = "هل يجوز فصل عامل تغيب عن العمل بدون انذار؟"
docs = vectordb.similarity_search(question, k=3)
len(docs)

In [ ]:
docs[0].page_content[:500]

In [ ]:
question = "هل يجوز فصل عامل تغيب عن العمل بدون انذار؟"  # "can an employer dismiss a worker who was absent without notice?"
docs = vectordb.similarity_search(question, k=3)

for d in docs:
    print(d.metadata)
    print(d.page_content[:300])
    print("---")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree('/content/chroma_v2', '/content/drive/MyDrive/law_chatbot_chroma_v2', dirs_exist_ok=True)
print("Vectorstore backed up to Drive as law_chatbot_chroma_v2 (separate from the original)")
